# An Intelligent Academic Assistant for Document-Based Question Answering
### IE624 · Retrieval-Augmented Generation (RAG) · Apurbo Talukder (25M1540), Sunil Kumar (25M1539)

This notebook implements the pipeline proposed in the pre-project report, phase by phase (Table 5):

| Phase | Activity | Section below |
|---|---|---|
| 1 | Upload ONE PDF (single-document mode) | §1 |
| 2 | Extract text with page metadata | §2 |
| 3 | Chunk (300 words / 50 overlap) + embed (all-MiniLM-L6-v2) | §3 |
| 4 | Semantic retrieval (cosine, top-3) | §4 |
| 5 | Suggested questions + answer generation (FLAN-T5) + citations from metadata | §5 |
| 6 | Chat interface (Gradio) + notebook fallback | §6 |
| 7 | Evaluation: RAG vs no-retrieval baseline | §7 |
| 8 | Reproducibility record | §8 |

**Single-document mode:** you upload one PDF and every question is answered from it. After indexing, the assistant suggests questions you can ask about that PDF (clickable in the app).

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ T4 GPU* (CPU also works, generation is just slower).

## 0. Setup

In [ ]:
!pip install -q pymupdf sentence-transformers transformers accelerate gradio

The three cells below write the project modules to disk (the same files are in the submitted code folder).

In [ ]:
%%writefile rag_core.py
"""
rag_core.py — An Intelligent Academic Assistant for Document-Based Question Answering
IE624 project: Retrieval-Augmented Generation (RAG) pipeline.

Pipeline (matches Section 4 of the pre-project report):

  OFFLINE INDEXING                         ONLINE QUESTION ANSWERING
  A. PDF collection                        F. User question
  B. Text extraction (page metadata)       G. Query embedding + top-k cosine search
  C. Cleaning + overlapping chunking       H. Evidence-first prompt construction
  D. Embedding generation                  I. Answer generation (pretrained model)
  E. Index storage (vectors + metadata)    J. Answer + citations from stored metadata

Single-document mode: one PDF is uploaded and all questions are answered from it
(set_single_pdf). After indexing, suggested questions about the PDF are generated
(suggest_questions), NotebookLM-style.

Libraries: PyMuPDF (fitz), sentence-transformers, NumPy, Hugging Face Transformers.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np

# --------------------------------------------------------------------------------------
# Configuration (all experiment settings in one place, for reproducibility — §5.7)
# --------------------------------------------------------------------------------------


@dataclass
class RAGConfig:
    # Embedding model (§5.3)
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    # Chunking (§5.3): ~300 words with 50 words overlap
    chunk_size_words: int = 300
    chunk_overlap_words: int = 50
    min_chunk_words: int = 40          # trailing fragments shorter than this are merged
    # Retrieval (§5.4)
    top_k: int = 3
    # Abstention gate: if the best cosine similarity is below this, the assistant says the
    # documents do not contain enough information WITHOUT calling the generator.
    # 0.30 is a starting value for all-MiniLM-L6-v2; tune it on the evaluation set.
    min_similarity: float = 0.30
    # Generator (§5.5). Default is a text-to-text model, as in the report.
    # Alternative (stronger, still free): "Qwen/Qwen2.5-1.5B-Instruct" (decoder/chat model).
    generator_model: str = "google/flan-t5-base"
    max_input_tokens: int = 1024       # question + context are truncated to this
    max_new_tokens: int = 200
    # Decoding: greedy (deterministic) so baseline and RAG are compared fairly (§6.2)
    num_beams: int = 1
    do_sample: bool = False
    repetition_penalty: float = 1.15
    # Suggested questions shown after the PDF is uploaded (NotebookLM-style)
    n_suggestions: int = 8


INSUFFICIENT_EVIDENCE = (
    "The available documents do not contain enough information to answer this question."
)

# --------------------------------------------------------------------------------------
# B. Text extraction
# --------------------------------------------------------------------------------------


def extract_pages(pdf_path: str | Path) -> List[Dict]:
    """Extract text page by page, keeping filename and 1-based page number.

    Uses PyMuPDF (fitz) as planned in §5.1; falls back to pypdf if PyMuPDF is missing.
    """
    pdf_path = Path(pdf_path)
    pages: List[Dict] = []
    try:
        import fitz  # PyMuPDF

        with fitz.open(pdf_path) as doc:
            for i, page in enumerate(doc):
                # sort=True orders text blocks top-to-bottom, left-to-right
                pages.append({"source": pdf_path.name, "page": i + 1,
                              "text": page.get_text("text", sort=True)})
    except ImportError:
        from pypdf import PdfReader

        reader = PdfReader(str(pdf_path))
        for i, page in enumerate(reader.pages):
            pages.append({"source": pdf_path.name, "page": i + 1,
                          "text": page.extract_text() or ""})
    return pages


# --------------------------------------------------------------------------------------
# C. Cleaning and chunking
# --------------------------------------------------------------------------------------

_LIGATURES = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi",
              "ﬄ": "ffl", "­": "", "​": ""}


def clean_text(text: str) -> str:
    """Conservative cleaning (§5.2): fixes extraction noise, keeps punctuation/equations."""
    for bad, good in _LIGATURES.items():
        text = text.replace(bad, good)
    # Re-join words hyphenated across a line break: "classi-\nfication" -> "classification"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    lines = []
    for line in text.splitlines():
        s = line.strip()
        # Drop lines that are only a page number (e.g. "12", "- 12 -", "Page 12")
        if re.fullmatch(r"(page\s*)?[-–]?\s*\d{1,4}\s*[-–]?", s, flags=re.I):
            continue
        lines.append(s)
    text = " ".join(lines)
    return re.sub(r"\s+", " ", text).strip()


def is_low_quality(text: str) -> bool:
    """Flag pages that are probably scanned/unreadable (little or garbled text)."""
    if len(text) < 30:
        return True
    letters = sum(c.isalpha() for c in text)
    return letters / max(len(text), 1) < 0.4


@dataclass
class Chunk:
    chunk_id: int
    source: str
    page_start: int
    page_end: int
    text: str
    n_words: int

    @property
    def page_label(self) -> str:
        if self.page_start == self.page_end:
            return f"p. {self.page_start}"
        return f"pp. {self.page_start}–{self.page_end}"


def chunk_document(pages: List[Dict], size: int, overlap: int, min_words: int,
                   start_id: int = 0) -> List[Chunk]:
    """Split one document into overlapping word windows.

    Words are tagged with their page number, so a chunk that crosses a page boundary
    keeps an exact page range (e.g. pp. 4–5) — citations come from metadata, never from
    the model (§4.3).
    """
    if overlap >= size:
        raise ValueError("chunk overlap must be smaller than chunk size")
    words, word_pages = [], []
    for p in pages:
        w = clean_text(p["text"]).split()
        words.extend(w)
        word_pages.extend([p["page"]] * len(w))
    if not words:
        return []

    source = pages[0]["source"]
    step = size - overlap
    starts = list(range(0, max(len(words) - overlap, 1), step))
    chunks: List[Chunk] = []
    for n, s in enumerate(starts):
        e = min(s + size, len(words))
        # Merge a short trailing fragment into the previous chunk instead of indexing it alone
        if n > 0 and (e - s) < min_words and chunks:
            prev = chunks[-1]
            extra = words[prev_end:e]
            if extra:
                prev.text += " " + " ".join(extra)
                prev.n_words += len(extra)
                prev.page_end = word_pages[e - 1]
            break
        chunks.append(Chunk(chunk_id=start_id + len(chunks), source=source,
                            page_start=word_pages[s], page_end=word_pages[e - 1],
                            text=" ".join(words[s:e]), n_words=e - s))
        prev_end = e
        if e == len(words):
            break
    return chunks


# --------------------------------------------------------------------------------------
# D + E. Embeddings and index storage
# --------------------------------------------------------------------------------------

_MODEL_CACHE: Dict[str, object] = {}


def get_embedder(model_name: str):
    if model_name not in _MODEL_CACHE:
        from sentence_transformers import SentenceTransformer

        _MODEL_CACHE[model_name] = SentenceTransformer(model_name)
    return _MODEL_CACHE[model_name]


def embed_texts(texts: List[str], model_name: str, batch_size: int = 32) -> np.ndarray:
    """Return L2-normalised embeddings, so cosine similarity = dot product."""
    model = get_embedder(model_name)
    vecs = model.encode(texts, batch_size=batch_size, show_progress_bar=len(texts) > 64,
                        convert_to_numpy=True, normalize_embeddings=True)
    return vecs.astype(np.float32)


def _file_hash(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


class DocumentIndex:
    """Vectors + chunk text + source metadata (stage E)."""

    def __init__(self, config: RAGConfig):
        self.config = config
        self.chunks: List[Chunk] = []
        self.embeddings: Optional[np.ndarray] = None
        self.documents: List[Dict] = []     # per-file report (pages, skipped pages, hash)
        self.folder: Optional[Path] = None  # where the index is saved (suggestions cache)

    # ---- building -------------------------------------------------------------------
    def build(self, pdf_paths: List[str | Path], verbose: bool = True) -> "DocumentIndex":
        cfg = self.config
        self.chunks, self.documents = [], []
        for path in sorted(Path(p) for p in pdf_paths):
            pages = extract_pages(path)
            good = [p for p in pages if not is_low_quality(clean_text(p["text"]))]
            skipped = [p["page"] for p in pages if p not in good]
            doc_chunks = chunk_document(good, cfg.chunk_size_words, cfg.chunk_overlap_words,
                                        cfg.min_chunk_words, start_id=len(self.chunks))
            self.chunks.extend(doc_chunks)
            self.documents.append({"source": path.name, "pages": len(pages),
                                   "skipped_pages": skipped, "chunks": len(doc_chunks),
                                   "md5": _file_hash(path)})
            if verbose:
                note = f" (skipped low-text pages: {skipped})" if skipped else ""
                print(f"  {path.name}: {len(pages)} pages -> {len(doc_chunks)} chunks{note}")
        if not self.chunks:
            raise ValueError("No extractable text found in the supplied PDFs.")
        t0 = time.time()
        self.embeddings = embed_texts([c.text for c in self.chunks], cfg.embedding_model)
        if verbose:
            print(f"Embedded {len(self.chunks)} chunks with {cfg.embedding_model} "
                  f"in {time.time() - t0:.1f}s -> matrix {self.embeddings.shape}")
        return self

    # ---- persistence ----------------------------------------------------------------
    def save(self, folder: str | Path) -> None:
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        self.folder = folder
        stale = folder / "suggestions.json"      # suggestions belong to the old index
        if stale.exists():
            stale.unlink()
        np.save(folder / "embeddings.npy", self.embeddings)
        with open(folder / "chunks.json", "w", encoding="utf-8") as f:
            json.dump([asdict(c) for c in self.chunks], f, ensure_ascii=False, indent=1)
        with open(folder / "index_info.json", "w", encoding="utf-8") as f:
            json.dump({"config": asdict(self.config), "documents": self.documents}, f, indent=2)

    @classmethod
    def load(cls, folder: str | Path, config: Optional[RAGConfig] = None) -> "DocumentIndex":
        folder = Path(folder)
        info = json.loads((folder / "index_info.json").read_text(encoding="utf-8"))
        saved = RAGConfig(**info["config"])
        if config is None:
            config = saved
        elif config.embedding_model != saved.embedding_model:
            raise ValueError("Index was built with a different embedding model; rebuild it.")
        idx = cls(config)
        idx.embeddings = np.load(folder / "embeddings.npy")
        idx.chunks = [Chunk(**c) for c in
                      json.loads((folder / "chunks.json").read_text(encoding="utf-8"))]
        idx.documents = info["documents"]
        idx.folder = folder
        return idx

    # ---- G. retrieval ---------------------------------------------------------------
    def search(self, query: str, k: Optional[int] = None) -> List[Dict]:
        """Embed the question and rank all chunks by cosine similarity; return top-k."""
        k = k or self.config.top_k
        q = embed_texts([query], self.config.embedding_model)[0]
        scores = self.embeddings @ q                      # cosine (vectors are normalised)
        k = min(k, len(scores))
        top = np.argpartition(-scores, k - 1)[:k]
        top = top[np.argsort(-scores[top])]
        return [{"rank": r + 1, "score": float(scores[i]), "chunk": self.chunks[i]}
                for r, i in enumerate(top)]


# --------------------------------------------------------------------------------------
# H. Prompt construction (§5.5)
# --------------------------------------------------------------------------------------


def build_rag_prompt(question: str, hits: List[Dict]) -> str:
    context = "\n\n".join(
        f"[{h['rank']}] ({h['chunk'].source}, {h['chunk'].page_label}) {h['chunk'].text}"
        for h in hits)
    # Question is placed BEFORE the context so that, if the input is truncated,
    # only the tail of the context is lost — never the question or the instructions.
    return (
        "Role: You are an academic study assistant.\n"
        "Task: Answer the user's question using only the provided context.\n"
        "Limitations: If the context does not support an answer, reply exactly: "
        f"\"{INSUFFICIENT_EVIDENCE}\"\n"
        "Style: Give a concise, clear explanation in complete sentences. "
        "Do not invent citations.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )


def build_baseline_prompt(question: str) -> str:
    """No-retrieval baseline (§6.2): same model, same decoding, no document context."""
    return (
        "Role: You are an academic study assistant.\n"
        "Task: Answer the user's question. If you do not know the answer, reply exactly: "
        f"\"{INSUFFICIENT_EVIDENCE}\"\n"
        "Style: Give a concise, clear explanation in complete sentences.\n\n"
        f"Question: {question}\n\nAnswer:"
    )


# --------------------------------------------------------------------------------------
# I. Answer generation
# --------------------------------------------------------------------------------------


class Generator:
    """Wraps either a text-to-text model (FLAN-T5) or a small chat model (e.g. Qwen2.5)."""

    def __init__(self, config: RAGConfig):
        import torch
        from transformers import (AutoConfig, AutoModelForCausalLM,
                                  AutoModelForSeq2SeqLM, AutoTokenizer)

        self.config = config
        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        name = config.generator_model
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.is_seq2seq = AutoConfig.from_pretrained(name).is_encoder_decoder
        dtype = torch.float16 if self.device == "cuda" and not self.is_seq2seq else torch.float32
        loader = AutoModelForSeq2SeqLM if self.is_seq2seq else AutoModelForCausalLM
        self.model = loader.from_pretrained(name, torch_dtype=dtype).to(self.device).eval()

    def generate(self, prompt: str, max_new_tokens: Optional[int] = None) -> str:
        cfg, tok, torch = self.config, self.tokenizer, self.torch
        if not self.is_seq2seq and getattr(tok, "chat_template", None):
            prompt = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                             tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", truncation=True,
                     max_length=cfg.max_input_tokens).to(self.device)
        gen_kwargs = dict(max_new_tokens=max_new_tokens or cfg.max_new_tokens,
                          num_beams=cfg.num_beams,
                          do_sample=cfg.do_sample, repetition_penalty=cfg.repetition_penalty)
        if tok.pad_token_id is None:
            gen_kwargs["pad_token_id"] = tok.eos_token_id
        with torch.no_grad():
            out = self.model.generate(**inputs, **gen_kwargs)
        if not self.is_seq2seq:                     # strip the echoed prompt
            out = out[:, inputs["input_ids"].shape[1]:]
        return tok.decode(out[0], skip_special_tokens=True).strip()


_ABSTAIN_PATTERNS = [
    "do not contain enough information", "does not contain enough information",
    "not enough information", "cannot be answered", "can't be answered",
    "not mentioned in the context", "context does not", "no information",
    "i don't know", "i do not know", "unanswerable",
]


def is_abstention(answer: str) -> bool:
    a = answer.lower()
    return any(p in a for p in _ABSTAIN_PATTERNS)


# --------------------------------------------------------------------------------------
# Assistant: F → J end to end
# --------------------------------------------------------------------------------------


@dataclass
class Answer:
    question: str
    answer: str
    abstained: bool
    sources: List[Dict] = field(default_factory=list)   # citations from stored metadata
    best_score: float = 0.0
    seconds: float = 0.0
    mode: str = "rag"


class AcademicAssistant:
    def __init__(self, index: Optional[DocumentIndex], config: RAGConfig,
                 generator: Optional[Generator] = None):
        self.index = index
        self.config = config
        self._generator = generator

    @property
    def generator(self) -> Generator:
        if self._generator is None:
            self._generator = Generator(self.config)
        return self._generator

    def ask(self, question: str, k: Optional[int] = None,
            min_similarity: Optional[float] = None) -> Answer:
        """RAG answer with citations. Response time covers retrieval + generation."""
        t0 = time.time()
        k = k or self.config.top_k
        thr = self.config.min_similarity if min_similarity is None else min_similarity
        hits = self.index.search(question, k)
        best = hits[0]["score"] if hits else 0.0
        sources = [{"rank": h["rank"], "score": round(h["score"], 3),
                    "source": h["chunk"].source, "pages": h["chunk"].page_label,
                    "page_start": h["chunk"].page_start, "page_end": h["chunk"].page_end,
                    "text": h["chunk"].text} for h in hits]

        if best < thr:      # evidence gate: nothing relevant retrieved → abstain
            return Answer(question, INSUFFICIENT_EVIDENCE, True, sources, best,
                          time.time() - t0)

        text = self.generator.generate(build_rag_prompt(question, hits))
        abstained = is_abstention(text) or not text
        if abstained:
            text = INSUFFICIENT_EVIDENCE
        return Answer(question, text, abstained, sources, best, time.time() - t0)

    def ask_baseline(self, question: str) -> Answer:
        t0 = time.time()
        text = self.generator.generate(build_baseline_prompt(question))
        return Answer(question, text, is_abstention(text), [], 0.0, time.time() - t0,
                      mode="baseline")

    # ---- suggested questions (NotebookLM-style) ---------------------------------------
    def suggested_questions(self, n: Optional[int] = None, refresh: bool = False,
                            seed: int = 0) -> List[str]:
        """Questions generated from the uploaded documents, cached in the index folder."""
        n = n or self.config.n_suggestions
        cache = self.index.folder / "suggestions.json" if self.index.folder else None
        if cache and cache.exists() and not refresh:
            cached = json.loads(cache.read_text(encoding="utf-8"))
            if len(cached) >= n:
                self._shown_suggestions = cached[:n]
                return cached[:n]
        shown = getattr(self, "_shown_suggestions", []) if refresh else []
        qs = suggest_questions(self.index, self.generator, n=n, seed=seed,
                               min_similarity=self.config.min_similarity, exclude=shown)
        if not qs:                         # nothing new left in a short PDF → reuse list
            qs = shown[-n:] if shown else suggest_questions(
                self.index, self.generator, n=n, min_similarity=self.config.min_similarity)
        self._shown_suggestions = shown + [q for q in qs if q not in shown]
        if cache:
            cache.write_text(json.dumps(qs, ensure_ascii=False, indent=1), encoding="utf-8")
        return qs



# --------------------------------------------------------------------------------------


# --------------------------------------------------------------------------------------
# Suggested questions
# --------------------------------------------------------------------------------------

_NOISE_PATTERNS = re.compile(
    r"(et al\.|doi|https?://|www\.|\.{4,}|internet source|student paper|similarity index|"
    r"submission id|<1%|isbn|vol\.|pp\.\s*\d)", re.I)


def is_contentful(text: str) -> bool:
    """Skip chunks that make poor question sources: references, table of contents,
    plagiarism-report pages, tables of numbers."""
    words = text.split()
    if len(words) < 80:
        return False
    letters = sum(ch.isalpha() for ch in text)
    if letters / max(len(text), 1) < 0.65:
        return False
    return len(_NOISE_PATTERNS.findall(text)) / len(words) < 0.02


def pick_representative_chunks(index: DocumentIndex, n: int, seed: int = 0,
                               diversity: float = 0.6) -> List[Chunk]:
    """Choose chunks that are typical of their document but different from each other
    (maximal marginal relevance), spread across documents in turn.
    seed=0 gives the first n in that order, seed=1 the next n, and so on (wrapping)."""
    order = _mmr_order(index, n * (seed + 1), diversity)
    if not order:
        return []
    start = (n * seed) % len(order)
    return (order[start:] + order[:start])[:n]


def _mmr_order(index: DocumentIndex, n: int, diversity: float) -> List[Chunk]:
    cand = [c for c in index.chunks if is_contentful(c.text)] or list(index.chunks)
    by_doc: Dict[str, List[Chunk]] = {}
    for c in cand:
        by_doc.setdefault(c.source, []).append(c)
    E = index.embeddings
    picked: List[Chunk] = []
    pools = {}
    for doc, cs in by_doc.items():
        ids = np.array([c.chunk_id for c in cs])
        centroid = E[ids].mean(0)
        centroid /= np.linalg.norm(centroid) + 1e-9
        rep = E[ids] @ centroid
        pools[doc] = (list(ids), list(rep))
    docs = list(pools)
    while len(picked) < n and any(pools[d][0] for d in docs):
        for d in docs:
            ids, rep = pools[d]
            if not ids or len(picked) >= n:
                continue
            if picked:
                sel = np.array([c.chunk_id for c in picked])
                redund = (E[ids] @ E[sel].T).max(1)
            else:
                redund = np.zeros(len(ids))
            score = (1 - diversity) * np.array(rep) - diversity * redund
            j = int(np.argmax(score))
            picked.append(index.chunks[ids[j]])
            ids.pop(j), rep.pop(j)
    return picked


# Question styles are rotated so the suggestion list mixes question types, like the
# evaluation set (factual / conceptual / comparison / findings).
QUESTION_STYLES = [
    "a factual question (what / which / how many) about a specific definition, number, "
    "method or result",
    "a conceptual question (why or how) about the reasoning, mechanism or purpose behind "
    "an idea",
    "a question comparing two concepts, methods, cases or results discussed in the text "
    "(if only one is discussed, ask how it works instead)",
    "a question about a finding, conclusion, assumption or limitation stated in the text",
]

QUESTION_PROMPT = (
    "Read the text below from an academic document. Write ONE clear question that a "
    "student could ask: {style}. The question must be answerable from the text, "
    "be specific (mention the concept, method or result by name), and must not refer to "
    "\"the text\" or \"the passage\". Reply with the question only.\n\n"
    "Text: {text}\n\nQuestion:"
)


def _clean_question(raw: str) -> Optional[str]:
    for line in raw.strip().splitlines():
        line = re.sub(r"^\s*(question\s*\d*\s*[:.-]|\d+[.)]|[-*•])\s*", "", line,
                      flags=re.I).strip().strip('"').strip()
        if not line:
            continue
        if "?" in line:
            line = line[: line.index("?") + 1]
        elif re.match(r"(explain|describe|discuss|summari[sz]e|compare|list|outline|define)\b",
                      line, re.I):
            line = line.rstrip(".")
        else:
            continue
        n_words = len(line.split())
        if 4 <= n_words <= 30 and not re.search(r"\b(the|this) (text|passage|excerpt)\b",
                                                 line, re.I):
            return line[0].upper() + line[1:]
    return None


def suggest_questions(index: DocumentIndex, generator: "Generator", n: int = 6,
                      seed: int = 0, min_similarity: float = 0.3,
                      exclude: Optional[List[str]] = None) -> List[str]:
    """Generate n clickable starter questions from the indexed documents.

    1. pick 2n representative, diverse, content-rich chunks;
    2. ask the generator for one question per chunk;
    3. keep a question only if retrieval for it finds its source passage in the top 3
       (or the best similarity ≥ min_similarity), and it is not a near-duplicate.
    A broad overview question is always offered first.
    """
    overview = ("What is the main objective and key contribution of this document?"
                if len(index.documents) == 1 else
                "What are the main topics covered in these documents?")
    questions: List[str] = []
    q_vecs: List[np.ndarray] = []
    if seed == 0:       # offer the overview question only if it can be answered from the PDF
        ov = embed_texts([overview], index.config.embedding_model)[0]
        if float((index.embeddings @ ov).max()) >= min_similarity:
            questions.append(overview)
    if exclude:                              # questions already shown → don't repeat them
        q_vecs = list(embed_texts(list(exclude), index.config.embedding_model))
    for attempt, chunk in enumerate(pick_representative_chunks(index, 2 * n, seed=seed)):
        if len(questions) >= n:
            break
        text = " ".join(chunk.text.split()[:350])
        style = QUESTION_STYLES[(attempt + seed) % len(QUESTION_STYLES)]
        q = _clean_question(generator.generate(QUESTION_PROMPT.format(style=style, text=text),
                                               max_new_tokens=60))
        if not q:
            continue
        v = embed_texts([q], index.config.embedding_model)[0]
        if q_vecs and max(float(v @ u) for u in q_vecs) > 0.85:      # near-duplicate
            continue
        # keep only questions the retriever can answer from this PDF: its source passage
        # must come back in the top 3, or the best match must pass the answer threshold
        scores = index.embeddings @ v
        top3 = np.argsort(-scores)[:3]
        if chunk.chunk_id not in top3 and float(scores.max()) < min_similarity:
            continue
        q_vecs.append(v)
        questions.append(q)
    fallbacks = ["What methodology or approach is used?", "What are the main findings or results?",
                 "What conclusions and limitations are discussed?",
                 "What are the key terms and definitions introduced?"]
    for f in fallbacks:
        if len(questions) >= n:
            break
        if f not in (exclude or []):
            questions.append(f)
    return questions[:n]


# --------------------------------------------------------------------------------------
# J. Presentation helpers
# --------------------------------------------------------------------------------------

def format_sources_markdown(ans: Answer, excerpt_words: int = 70) -> str:
    if not ans.sources:
        return "_No sources (baseline mode)._"
    lines = []
    if ans.abstained:
        lines.append(f"> ⚠️ Best match similarity was {ans.best_score:.2f}. The closest "
                     "passages are shown below, but they were judged insufficient.\n")
    for s in ans.sources:
        words = s["text"].split()
        excerpt = " ".join(words[:excerpt_words]) + (" …" if len(words) > excerpt_words else "")
        lines.append(f"**[{s['rank']}] {s['source']} — {s['pages']}** "
                     f"(similarity {s['score']:.2f})\n\n> {excerpt}\n")
    lines.append(f"_Response time: {ans.seconds:.2f} s_")
    return "\n".join(lines)


def format_answer_with_citations(ans: Answer) -> str:
    if ans.abstained or not ans.sources:
        return ans.answer
    cites = "; ".join(f"[{s['rank']}] {s['source']}, {s['pages']}" for s in ans.sources)
    return f"{ans.answer}\n\nSources: {cites}"


def set_single_pdf(pdf_path: str | Path, pdf_dir: str | Path = "documents",
                   index_dir: str | Path = "index") -> Path:
    """Single-document mode: empty the documents folder (and the old index), then copy in
    the one PDF to be questioned. Returns the path of the copied PDF."""
    import shutil

    src = Path(pdf_path)
    if src.suffix.lower() != ".pdf":
        raise ValueError(f"{src.name} is not a PDF file.")
    pdf_dir = Path(pdf_dir)
    shutil.rmtree(pdf_dir, ignore_errors=True)
    shutil.rmtree(index_dir, ignore_errors=True)
    pdf_dir.mkdir(parents=True, exist_ok=True)
    # tidy name: "thesis.pdf.pdf" -> "thesis.pdf"
    name = src.name
    while name.lower().endswith(".pdf.pdf"):
        name = name[:-4]
    dst = pdf_dir / name
    shutil.copy(src, dst)
    return dst


def list_pdfs(folder: str | Path) -> List[Path]:
    return sorted(p for p in Path(folder).glob("**/*") if p.suffix.lower() == ".pdf")


def build_or_load_index(pdf_folder: str | Path, index_folder: str | Path,
                        config: RAGConfig, rebuild: bool = False) -> DocumentIndex:
    """Reuse a saved index if the PDFs and chunk settings are unchanged; otherwise rebuild."""
    pdfs = list_pdfs(pdf_folder)
    if not pdfs:
        raise FileNotFoundError(f"No PDF files found in {pdf_folder}")
    index_folder = Path(index_folder)
    if not rebuild and (index_folder / "index_info.json").exists():
        info = json.loads((index_folder / "index_info.json").read_text())
        same_files = {d["source"]: d["md5"] for d in info["documents"]} == \
                     {p.name: _file_hash(p) for p in pdfs}
        saved = info["config"]
        same_cfg = all(saved.get(k) == getattr(config, k) for k in
                       ("embedding_model", "chunk_size_words", "chunk_overlap_words",
                        "min_chunk_words"))
        if same_files and same_cfg:
            print(f"Loaded existing index from {index_folder}")
            return DocumentIndex.load(index_folder, config)
    print(f"Building index from {len(pdfs)} PDF(s):")
    idx = DocumentIndex(config).build(pdfs)
    idx.save(index_folder)
    return idx

In [ ]:
%%writefile evaluate.py
"""
evaluate.py — Evaluation of the RAG assistant vs. a no-retrieval baseline (§6 of the report).

Workflow
  1. Fill in eval_questions.csv (≈25 questions: 8 factual, 7 conceptual, 5 comparison,
     5 unanswerable) with expected answers and gold source/page.
  2. run_evaluation(...)  -> writes results_raw.csv  (answers, retrieved sources, timings,
     automatic metrics, and EMPTY columns for manual scoring).
  3. Open results_raw.csv, fill the manual rubric columns (0/1/2) and citation_ok (0/1),
     save as results_scored.csv.
  4. summarise(...)  -> the metrics table for the final report.

Automatic metrics: Recall@k, abstention success, response time, citation match (gold
source/page among cited chunks), token-F1 vs. expected answer (auxiliary only).
Manual metrics: answer correctness (0–2), faithfulness (0–2), citation accuracy (0/1).
"""

from __future__ import annotations

import re
import string
from collections import Counter
from pathlib import Path
from typing import List, Optional

import pandas as pd

from rag_core import AcademicAssistant

QUESTION_COLUMNS = ["id", "question", "type", "answerable", "expected_answer",
                    "source_file", "source_pages"]


# ---------------------------------------------------------------------------- helpers

def _yes(v) -> bool:
    return str(v).strip().lower() in {"yes", "y", "true", "1"}


def parse_pages(v) -> List[int]:
    """'4' -> [4];  '4;5' or '4,5' -> [4,5];  '4-6' -> [4,5,6];  blank -> []"""
    if pd.isna(v) or str(v).strip() == "":
        return []
    pages = []
    for part in re.split(r"[;,]", str(v)):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-", 1)
            pages.extend(range(int(float(a)), int(float(b)) + 1))
        elif part:
            pages.append(int(float(part)))
    return pages


def _gold_hit(src: dict, gold_file: str, gold_pages: List[int]) -> bool:
    """A retrieved chunk is relevant if it comes from the gold file and its page range
    covers at least one gold page (or any page, if no gold page was given)."""
    def stem(name: str) -> str:
        name = str(name).strip().lower()
        while name.endswith(".pdf"):
            name = name[:-4]
        return name

    gold_files = [stem(g) for g in str(gold_file).split(";") if g.strip()]
    s = stem(src["source"])
    # exact match, or one name contains the other (handles renamed copies / ".pdf.pdf")
    if not any(g == s or (g and (g in s or s in g)) for g in gold_files):
        return False
    if not gold_pages:
        return True
    return any(src["page_start"] <= p <= src["page_end"] for p in gold_pages)


def _normalise(text: str) -> List[str]:
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return text.split()


def token_f1(pred: str, gold: str) -> float:
    """SQuAD-style token overlap F1 — a rough automatic proxy, not a replacement for the
    manual correctness rubric."""
    p, g = _normalise(pred), _normalise(gold)
    if not p or not g:
        return 0.0
    common = Counter(p) & Counter(g)
    same = sum(common.values())
    if same == 0:
        return 0.0
    prec, rec = same / len(p), same / len(g)
    return 2 * prec * rec / (prec + rec)


# ---------------------------------------------------------------------------- run

_ALIASES = {
    "question": ["q", "questions", "query", "question_text"],
    "type": ["category", "question_type", "qtype", "kind"],
    "answerable": ["is_answerable", "answerable_yes_no", "answerable_y_n", "can_answer"],
    "expected_answer": ["answer", "expected", "gold_answer", "reference_answer",
                        "ground_truth", "correct_answer", "reference"],
    "source_file": ["source", "file", "document", "gold_source", "source_document",
                    "pdf", "filename", "doc"],
    "source_pages": ["page", "pages", "source_page", "gold_pages", "page_number",
                     "page_no", "gold_page"],
    "id": ["no", "sno", "s_no", "number", "qid", "question_id", "sr_no"],
}


def _normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    norm = {c: re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
            for c in df.columns}
    df = df.rename(columns=norm)
    for canon, alts in _ALIASES.items():
        if canon not in df.columns:
            for a in alts:
                if a in df.columns:
                    df = df.rename(columns={a: canon})
                    break
    return df


def load_questions(path: str | Path) -> pd.DataFrame:
    """Read the question sheet. Column names are matched loosely (e.g. "Answer", "Page",
    "Source" work). Only `question` is strictly required; missing optional columns are
    filled in with a warning (Recall@k needs source_file + source_pages)."""
    # utf-8-sig removes Excel's BOM; sep=None also accepts ;-separated Excel exports
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    df = _normalise_columns(df)
    if "question" not in df.columns:
        raise ValueError(f"No 'question' column found. Columns are: {list(df.columns)}. "
                         f"Expected: {QUESTION_COLUMNS}")
    if "id" not in df.columns:
        df.insert(0, "id", range(1, len(df) + 1))
    if "type" not in df.columns:
        df["type"] = "unspecified"
    df["type"] = df["type"].fillna("unspecified").astype(str).str.strip().str.lower()
    if "answerable" not in df.columns:
        df["answerable"] = df["type"].ne("unanswerable").map({True: "yes", False: "no"})
        print("ℹ️ No 'answerable' column: inferred from type (unanswerable → no).")
    for col in ("expected_answer", "source_file", "source_pages"):
        if col not in df.columns:
            df[col] = ""
            print(f"⚠️ No '{col}' column: filled blank."
                  + (" Recall@k cannot be computed without it." if col != "expected_answer"
                     else " Token-F1 cannot be computed without it."))
    df[["expected_answer", "source_file", "source_pages"]] = \
        df[["expected_answer", "source_file", "source_pages"]].fillna("")
    todo = df["question"].astype(str).str.strip().str.upper().str.startswith("TODO")
    if todo.any():
        print(f"Skipping {int(todo.sum())} unfilled template row(s) (question starts with TODO).")
        df = df[~todo].copy()
    df["answerable"] = df["answerable"].map(_yes)
    return df


def run_evaluation(assistant: AcademicAssistant, questions_csv: str | Path,
                   out_csv: str | Path = "results_raw.csv", k: Optional[int] = None,
                   run_baseline: bool = True) -> pd.DataFrame:
    qs = load_questions(questions_csv)
    k = k or assistant.config.top_k
    rows = []
    for _, q in qs.iterrows():
        print(f"[{q['id']}] {q['question'][:70]}")
        gold_pages = parse_pages(q["source_pages"])
        rag = assistant.ask(q["question"], k=k)

        has_gold = q["answerable"] and str(q["source_file"]).strip() != ""
        hits = [_gold_hit(s, q["source_file"], gold_pages) for s in rag.sources] \
            if has_gold else []
        row = {
            "id": q["id"], "type": q["type"], "answerable": q["answerable"],
            "question": q["question"], "expected_answer": q["expected_answer"],
            "gold_source": q["source_file"], "gold_pages": q["source_pages"],
            # --- RAG condition
            "rag_answer": rag.answer,
            "rag_abstained": rag.abstained,
            "rag_best_similarity": round(rag.best_score, 3),
            "rag_cited": " | ".join(f"{s['source']} {s['pages']}" for s in rag.sources),
            f"recall@{k}": (any(hits) if has_gold else None),
            "first_relevant_rank": (hits.index(True) + 1 if any(hits) else None),
            "rag_seconds": round(rag.seconds, 3),
            "rag_token_f1": (round(token_f1(rag.answer, str(q["expected_answer"])), 3)
                             if q["answerable"] and q["expected_answer"] else None),
            # --- manual scoring (fill in by hand)
            "rag_correctness_0to2": "", "rag_faithfulness_0to2": "", "rag_citation_ok_0or1": "",
        }
        if run_baseline:
            base = assistant.ask_baseline(q["question"])
            row.update({
                "baseline_answer": base.answer,
                "baseline_abstained": base.abstained,
                "baseline_seconds": round(base.seconds, 3),
                "baseline_token_f1": (round(token_f1(base.answer, str(q["expected_answer"])), 3)
                                      if q["answerable"] and q["expected_answer"] else None),
                "baseline_correctness_0to2": "",
            })
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {len(df)} rows to {out_csv}. Fill the *_0to2 / *_0or1 columns, then "
          "call summarise().")
    return df


# ---------------------------------------------------------------------------- summary

def _bool_mean(series) -> Optional[float]:
    if series is None:
        return None
    s = series[series.notna() & (series.astype(str).str.strip() != "")]
    return round(float(s.map(_yes).mean()), 3) if len(s) else None


def _mean(series) -> Optional[float]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    return round(float(s.mean()), 3) if len(s) else None


def _mrr(ans: pd.DataFrame, rcol: str) -> Optional[float]:
    if rcol not in ans.columns:
        return None
    gold = ans[ans[rcol].notna() & (ans[rcol].astype(str).str.strip() != "")]
    if not len(gold):
        return None
    rr = 1 / pd.to_numeric(gold["first_relevant_rank"], errors="coerce")
    return round(float(rr.fillna(0).mean()), 3)


def summarise(results: str | Path | pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """Build the metrics table (Table 3 of the report) for both conditions."""
    df = pd.read_csv(results) if not isinstance(results, pd.DataFrame) else results.copy()
    df["answerable"] = df["answerable"].map(_yes)
    ans, unans = df[df["answerable"]], df[~df["answerable"]]
    rcol = f"recall@{k}"
    has_base = "baseline_answer" in df.columns

    def abst_rate(col):
        if col not in df.columns or not len(unans):
            return None
        return round(unans[col].map(_yes).mean(), 3)

    def false_abst_rate(col):
        if col not in df.columns or not len(ans):
            return None
        return round(ans[col].map(_yes).mean(), 3)

    metrics = [
        (f"Recall@{k} (answerable)", _bool_mean(ans[rcol]) if rcol in df else None, "—"),
        ("Mean reciprocal rank of first relevant chunk", _mrr(ans, rcol), "—"),
        ("Answer correctness, mean 0–2 (manual)", _mean(ans.get("rag_correctness_0to2")),
         _mean(ans.get("baseline_correctness_0to2")) if has_base else "—"),
        ("Faithfulness, mean 0–2 (manual)", _mean(ans.get("rag_faithfulness_0to2")), "—"),
        ("Citation accuracy (manual)", _mean(ans.get("rag_citation_ok_0or1")), "—"),
        ("Token-F1 vs expected answer (auto, auxiliary)", _mean(ans.get("rag_token_f1")),
         _mean(ans.get("baseline_token_f1")) if has_base else "—"),
        ("Abstention success (unanswerable)", abst_rate("rag_abstained"),
         abst_rate("baseline_abstained") if has_base else "—"),
        ("False abstention rate (answerable)", false_abst_rate("rag_abstained"),
         false_abst_rate("baseline_abstained") if has_base else "—"),
        ("Mean response time (s)", _mean(df["rag_seconds"]),
         _mean(df["baseline_seconds"]) if has_base else "—"),
    ]
    table = pd.DataFrame(metrics, columns=["Metric", "RAG", "Baseline (no retrieval)"])
    table = table.fillna("not scored yet")
    return table


def breakdown_by_type(results: str | Path | pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """Per question-type view (factual / conceptual / comparison / unanswerable)."""
    df = pd.read_csv(results) if not isinstance(results, pd.DataFrame) else results.copy()
    df["answerable"] = df["answerable"].map(_yes)
    out = []
    for t, g in df.groupby("type"):
        out.append({
            "type": t, "n": len(g),
            f"recall@{k}": _bool_mean(g[f"recall@{k}"]) if g["answerable"].any() else "—",
            "rag_correctness": _mean(g.get("rag_correctness_0to2")),
            "baseline_correctness": _mean(g.get("baseline_correctness_0to2")),
            "rag_abstained_%": round(100 * g["rag_abstained"].map(_yes).mean(), 1),
        })
    return pd.DataFrame(out).fillna("—")


def threshold_sweep(results: str | Path | pd.DataFrame,
                    thresholds=(0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5)) -> pd.DataFrame:
    """Shows how the similarity gate alone would trade off abstention on unanswerable
    questions against false abstention on answerable ones — use it to tune
    RAGConfig.min_similarity (report the chosen value)."""
    df = pd.read_csv(results) if not isinstance(results, pd.DataFrame) else results.copy()
    df["answerable"] = df["answerable"].map(_yes)
    rows = []
    for t in thresholds:
        gated = df["rag_best_similarity"] < t
        rows.append({"min_similarity": t,
                     "unanswerable_abstained": round(gated[~df["answerable"]].mean(), 3),
                     "answerable_wrongly_gated": round(gated[df["answerable"]].mean(), 3)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile app.py
"""
app.py — Gradio chat interface for the Intelligent Academic Assistant (§5.6).

Run:   python app.py --share            (in Colab, or call launch_app(assistant) from the notebook)

• Single-document mode: upload ONE PDF → the index is built automatically → suggested
  questions appear as clickable buttons (NotebookLM-style). Uploading another PDF
  replaces the previous one and starts a new chat. "↻ More suggestions" = fresh set.
• Source panel: document, page, similarity, supporting excerpt.
• Settings: top-k, abstention threshold, compare with no-retrieval baseline.
"""

from __future__ import annotations

import argparse
import shutil
from pathlib import Path

from rag_core import (AcademicAssistant, RAGConfig, build_or_load_index, list_pdfs,
                      format_answer_with_citations, format_sources_markdown, set_single_pdf)

MAX_SUGGESTIONS = 8                       # number of button slots in the UI
EMPTY_SOURCES = "_Sources for the latest answer appear here._"


def launch_app(assistant: AcademicAssistant, pdf_dir: str = "documents",
               index_dir: str = "index", share: bool = False):
    import gradio as gr

    state = {"seed": 0}

    # ------------------------------------------------------------------ helpers
    def doc_summary():
        if assistant.index is None:
            return "_No document yet. Upload a PDF above._"
        d = assistant.index.documents[0]
        cfg = assistant.config
        text = (f"📄 **{d['source']}** · {d['pages']} pages · {d['chunks']} chunks "
                f"(chunk {cfg.chunk_size_words} words, overlap {cfg.chunk_overlap_words})")
        if d["pages"] and len(d.get("skipped_pages", [])) > 0.3 * d["pages"]:
            text += ("\n\n⚠️ Most pages have no readable text (scanned or Turnitin copy?). "
                     "Upload the original PDF.")
        return text

    def suggestion_updates(questions):
        ups = [gr.update(value=q, visible=True) for q in questions[:MAX_SUGGESTIONS]]
        ups += [gr.update(value="", visible=False)] * (MAX_SUGGESTIONS - len(ups))
        return ups

    def load_suggestions(refresh=False):
        if assistant.index is None:
            return ["_Upload a PDF to see questions you can ask about it._"] + \
                suggestion_updates([])
        if refresh:
            state["seed"] += 1
        try:
            qs = assistant.suggested_questions(refresh=refresh, seed=state["seed"])
        except Exception as e:
            return [f"_Could not generate suggestions: {e}_"] + suggestion_updates([])
        return (["**💡 Questions you can ask about this PDF** — click one to ask it"]
                + suggestion_updates(qs))

    # ------------------------------------------------------------------ chat
    def respond(message, history, k, threshold, compare):
        history = list(history or [])
        if not message or not str(message).strip():
            return history, gr.update(), ""
        history.append({"role": "user", "content": message})
        if assistant.index is None:
            history.append({"role": "assistant",
                            "content": "Please upload a PDF first."})
            return history, EMPTY_SOURCES, ""
        ans = assistant.ask(message, k=int(k), min_similarity=float(threshold))
        reply = format_answer_with_citations(ans)
        if compare:
            base = assistant.ask_baseline(message)
            reply += f"\n\n---\n**Baseline (no retrieval):** {base.answer}"
        history.append({"role": "assistant", "content": reply})
        return history, format_sources_markdown(ans), ""

    # ------------------------------------------------------------------ documents
    def index_document(file, rebuild=False):
        """Upload → replace the current PDF → rebuild index → new suggestions, new chat."""
        try:
            if file is not None:
                path = file if isinstance(file, str) else file.name
                set_single_pdf(path, pdf_dir, index_dir)
            if not list_pdfs(pdf_dir):
                assistant.index = None
                return [doc_summary()] + load_suggestions() + [gr.update(), gr.update()]
            assistant.index = build_or_load_index(pdf_dir, index_dir, assistant.config,
                                                  rebuild=rebuild or file is not None)
        except Exception as e:
            return [f"❌ {e}"] + load_suggestions() + [gr.update(), gr.update()]
        state["seed"] = 0
        return [doc_summary()] + load_suggestions() + [[], EMPTY_SOURCES]

    def remove_document():
        shutil.rmtree(pdf_dir, ignore_errors=True)
        shutil.rmtree(index_dir, ignore_errors=True)
        assistant.index = None
        return [doc_summary()] + load_suggestions() + [[], EMPTY_SOURCES]

    # ------------------------------------------------------------------ layout
    try:
        chatbot = gr.Chatbot(type="messages", height=500, label="Academic Assistant")
    except TypeError:               # Gradio ≥6 uses the messages format by default
        chatbot = gr.Chatbot(height=500, label="Academic Assistant")

    with gr.Blocks(title="Academic Assistant (RAG)") as demo:
        gr.Markdown("## 📚 Intelligent Academic Assistant\n"
                    "Upload one PDF, then click a suggested question or type your own. "
                    "Answers are generated only from passages of that PDF and cite the page. "
                    "**Always check the cited passage.**")
        with gr.Row():
            with gr.Column(scale=3):
                upload = gr.File(file_count="single", file_types=[".pdf"],
                                 label="Upload ONE PDF (text-based) — indexing starts "
                                       "automatically; a new upload replaces the old one")
                docs_md = gr.Markdown(doc_summary())
                sugg_title = gr.Markdown()
                sugg_btns = []
                with gr.Row():
                    for i in range(MAX_SUGGESTIONS):
                        sugg_btns.append(gr.Button("", visible=False, size="sm",
                                                   variant="secondary", min_width=200))
                more_btn = gr.Button("↻ More suggestions", size="sm")
                chatbot.render()
                with gr.Row():
                    msg = gr.Textbox(placeholder="Ask a question about this PDF…",
                                     show_label=False, scale=5)
                    send = gr.Button("Ask", variant="primary", scale=1)
                clear = gr.Button("Clear chat")
            with gr.Column(scale=2):
                sources = gr.Markdown(EMPTY_SOURCES, label="Sources")

        with gr.Accordion("Settings", open=False):
            with gr.Row():
                k = gr.Slider(1, 8, value=assistant.config.top_k, step=1,
                              label="Top-k passages retrieved")
                thr = gr.Slider(0.0, 0.8, value=assistant.config.min_similarity, step=0.05,
                                label="Min. similarity to answer (abstention gate)")
                compare = gr.Checkbox(False, label="Also show no-retrieval baseline answer")
            clear_docs = gr.Button("🗑 Remove document", variant="stop")

        # ------------------------------------------------------------------ events
        chat_inputs = [chatbot, k, thr, compare]
        send.click(respond, [msg] + chat_inputs, [chatbot, sources, msg])
        msg.submit(respond, [msg] + chat_inputs, [chatbot, sources, msg])
        for b in sugg_btns:          # clicking a suggestion asks it (button value = question)
            b.click(respond, [b] + chat_inputs, [chatbot, sources, msg])
        more_btn.click(lambda: load_suggestions(refresh=True), None, [sugg_title] + sugg_btns)
        clear.click(lambda: ([], EMPTY_SOURCES), None, [chatbot, sources])

        doc_outputs = [docs_md, sugg_title] + sugg_btns + [chatbot, sources]
        upload.upload(index_document, [upload], doc_outputs)
        clear_docs.click(remove_document, None, doc_outputs)
        demo.load(lambda: load_suggestions(), None, [sugg_title] + sugg_btns)

    demo.queue().launch(share=share, debug=False)
    return demo


def console_loop(assistant: AcademicAssistant):
    """Notebook/terminal fallback if Gradio is unavailable (§5.6)."""
    print("Academic Assistant — type a question, a suggestion number, or 'quit'.")
    sugg = assistant.suggested_questions()
    for i, q in enumerate(sugg, 1):
        print(f"  {i}. {q}")
    while True:
        q = input("\nQuestion: ").strip()
        if q.lower() in {"quit", "exit", "q", ""}:
            break
        if q.isdigit() and 1 <= int(q) <= len(sugg):
            q = sugg[int(q) - 1]
            print("→", q)
        ans = assistant.ask(q)
        print("\nAnswer:", ans.answer)
        for s in ans.sources:
            print(f"  [{s['rank']}] {s['source']} {s['pages']}  (sim {s['score']:.2f})")
        print(f"  ({ans.seconds:.2f} s)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--pdf", help="the one PDF to ask questions about")
    ap.add_argument("--pdf_dir", default="documents")
    ap.add_argument("--index_dir", default="index")
    ap.add_argument("--generator", default=RAGConfig.generator_model)
    ap.add_argument("--rebuild", action="store_true")
    ap.add_argument("--share", action="store_true", help="public Gradio link (Colab)")
    ap.add_argument("--console", action="store_true", help="text loop instead of Gradio")
    args = ap.parse_args()

    cfg = RAGConfig(generator_model=args.generator)
    if args.pdf:
        set_single_pdf(args.pdf, args.pdf_dir, args.index_dir)
    idx = None
    if Path(args.pdf_dir).exists() and list_pdfs(args.pdf_dir):
        idx = build_or_load_index(args.pdf_dir, args.index_dir, cfg, rebuild=args.rebuild)
    assistant = AcademicAssistant(idx, cfg)
    if args.console:
        if idx is None:
            raise SystemExit(f"Put PDFs in {args.pdf_dir}/ first.")
        console_loop(assistant)
    else:
        launch_app(assistant, args.pdf_dir, args.index_dir, share=args.share)


In [ ]:
import importlib, rag_core, evaluate
importlib.reload(rag_core); importlib.reload(evaluate)
from rag_core import *
from evaluate import run_evaluation, summarise, breakdown_by_type, threshold_sweep
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1. Phase 1 — Upload ONE PDF
Single-document mode: upload one text-based PDF (lecture notes, a paper, a thesis). Every question is
answered from that PDF only. Running this cell again with a different PDF replaces the previous one.
Use the original PDF, not a scanned or Turnitin copy.


In [ ]:
import os
PDF_DIR, INDEX_DIR = "documents", "index"

try:                                   # Colab upload widget
    from google.colab import files
    uploaded = files.upload()          # choose ONE PDF
    pdfs = [n for n in uploaded if n.lower().endswith(".pdf")]
    if not pdfs:
        raise ValueError("Please upload a PDF file.")
    if len(pdfs) > 1:
        print(f"⚠️ Single-document mode: using only {pdfs[0]}")
    PDF_PATH = set_single_pdf(pdfs[0], PDF_DIR, INDEX_DIR)   # replaces any previous PDF
except ImportError:
    PDF_PATH = set_single_pdf("my_document.pdf", PDF_DIR, INDEX_DIR)   # local: set the path

# Alternative: a PDF on Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# PDF_PATH = set_single_pdf("/content/drive/MyDrive/thesis.pdf", PDF_DIR, INDEX_DIR)

print("Document:", PDF_PATH.name)


## Experiment configuration (all settings in one place)

In [ ]:
config = RAGConfig(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    chunk_size_words=300, chunk_overlap_words=50,
    top_k=3,
    min_similarity=0.30,                 # abstention gate; tuned in §7
    n_suggestions=8,                     # suggested questions shown after upload
    generator_model="google/flan-t5-base",   # alternatives: "google/flan-t5-large",
                                             # "Qwen/Qwen2.5-1.5B-Instruct"
)
config

## 2. Phase 2 — Text extraction with page metadata
Inspect the extraction before indexing (risk mitigation: *poor PDF extraction*). Pages with very little
or garbled text (likely scanned) are flagged and later skipped rather than adding OCR.

In [ ]:
rows = []
for pdf in list_pdfs(PDF_DIR):
    for p in extract_pages(pdf):
        t = clean_text(p["text"])
        rows.append({"source": p["source"], "page": p["page"], "words": len(t.split()),
                     "low_quality": is_low_quality(t), "preview": t[:100]})
pages_df = pd.DataFrame(rows)
display(pages_df.groupby("source").agg(pages=("page", "count"), words=("words", "sum"),
                                       low_quality_pages=("low_quality", "sum")))
pages_df.head(10)

## 3. Phase 3 — Chunking and embeddings → searchable index

In [ ]:
index = build_or_load_index(PDF_DIR, INDEX_DIR, config, rebuild=False)
chunks_df = pd.DataFrame([{"id": c.chunk_id, "source": c.source, "pages": c.page_label,
                           "words": c.n_words, "text": c.text[:150]} for c in index.chunks])
print(chunks_df["words"].describe().round(1).to_string())
chunks_df.head()

## 4. Phase 4 — Semantic retrieval (top-k cosine similarity)

In [ ]:
def show_retrieval(question, k=config.top_k):
    hits = index.search(question, k)
    return pd.DataFrame([{"rank": h["rank"], "similarity": round(h["score"], 3),
                          "source": h["chunk"].source, "pages": h["chunk"].page_label,
                          "excerpt": h["chunk"].text[:200]} for h in hits])

show_retrieval("What is the main objective and key contribution of this document?")

## 5. Phase 5 — Answer generation with citations
The prompt follows §5.5 (role, task, evidence, limitations, style). Citations are built from the stored
metadata of the retrieved chunks, **not** generated by the model. If the best similarity is below
`min_similarity`, or the model says the context is insufficient, the assistant abstains.

In [ ]:
assistant = AcademicAssistant(index, config)      # generator loads on first question
print("Generator device:", assistant.generator.device)

from IPython.display import Markdown
def ask(question, compare=False):
    ans = assistant.ask(question)
    out = f"**Q:** {question}\n\n**A:** {format_answer_with_citations(ans)}\n\n"
    if compare:
        out += f"**Baseline (no retrieval):** {assistant.ask_baseline(question).answer}\n\n"
    out += "---\n" + format_sources_markdown(ans)
    display(Markdown(out))


### Questions you can ask about this PDF (NotebookLM-style)
The code picks representative, varied passages from the PDF (maximal marginal relevance), and the model
writes one question per passage. Question types rotate between factual, conceptual (why/how),
comparison, and findings/limitations. A question is kept only if retrieval finds evidence for it.
The list is cached in the index folder.


In [ ]:
suggestions = assistant.suggested_questions()
print(f"Questions you can ask about {PDF_PATH.name}:\n")
for i, q in enumerate(suggestions, 1):
    print(f"{i}. {q}")


In [ ]:
# Ask a suggested question (index from 0), or type your own question as a string
ask(suggestions[0], compare=True)
# ask(suggestions[2])
# ask("Your own question here")


In [ ]:
# More questions (skips the ones already shown)
more = [q for q in assistant.suggested_questions(refresh=True, seed=1) if q not in suggestions]
if not more:
    print("No new questions found — this PDF is short; try the list above.")
for i, q in enumerate(more, len(suggestions) + 1):
    print(f"{i}. {q}")
suggestions += more


In [ ]:
# Prompt actually sent to the model (for the report / debugging)
q = "What is overfitting?"
print(build_rag_prompt(q, index.search(q, config.top_k))[:1500])

## 6. Phase 6 — Chat interface
The app opens with the PDF from §1 already loaded and shows **suggested questions as clickable buttons**:
click one to ask it, or type your own. "↻ More suggestions" gives a new set. Uploading a different PDF in
the app replaces the current one (indexing starts automatically and the chat restarts). The source panel
shows document, page, similarity and excerpt; *Settings* has top-k, the abstention threshold and a
*compare with baseline* toggle. With `share=True` Colab gives a public link.
If Gradio causes problems, run the fallback loop in the next cell instead.


In [ ]:
from app import launch_app
demo = launch_app(assistant, PDF_DIR, INDEX_DIR, share=True)

In [ ]:
# Fallback: simple notebook input loop (type 'quit' to stop)
# from app import console_loop; console_loop(assistant)

## 7. Phase 7 — Evaluation: RAG vs no-retrieval baseline
The questions must be about **the PDF uploaded in §1** (`source_file` = that PDF's name).
1. Copy `eval_questions_template.csv` → `eval_questions.csv` and fill in **25 questions**
   (8 factual, 7 conceptual, 5 comparison, 5 unanswerable) with expected answer, gold file and page(s).
   Page formats: `4`, `4;5` or `4-6`. Rows whose question still starts with `TODO` are skipped.
2. Run the cell below → `results_raw.csv`.
3. Score by hand in a spreadsheet: `rag_correctness_0to2`, `rag_faithfulness_0to2`,
   `rag_citation_ok_0or1`, `baseline_correctness_0to2`. Save as `results_scored.csv` and upload it.

In [ ]:
import os, shutil
if not os.path.exists("eval_questions.csv"):
    try:
        from google.colab import files
        print("Upload your filled eval_questions.csv"); files.upload()
    except ImportError:
        pass

results = run_evaluation(assistant, "eval_questions.csv", "results_raw.csv",
                         k=config.top_k, run_baseline=True)
results[["id", "type", "recall@3", "rag_best_similarity", "rag_abstained",
         "rag_answer", "baseline_answer"]]

In [ ]:
# Automatic metrics now; manual columns show 'not scored yet' until results_scored.csv exists
scored = "results_scored.csv" if os.path.exists("results_scored.csv") else "results_raw.csv"
display(summarise(scored, k=config.top_k))
display(breakdown_by_type(scored, k=config.top_k))

In [ ]:
# Tune the abstention gate: pick the threshold that abstains on unanswerable questions
# without blocking answerable ones; report the value you choose.
threshold_sweep("results_raw.csv")

In [ ]:
# Error analysis (§6.4): retrieval misses, wrong abstentions, and hallucination-prone cases
r = pd.read_csv("results_raw.csv")
print("Retrieval misses (gold passage not in top-k):")
display(r[(r["answerable"]) & (r["recall@3"] == False)][["id", "question", "gold_source",
                                                         "gold_pages", "rag_cited"]])
print("Answerable but abstained:")
display(r[(r["answerable"]) & (r["rag_abstained"])][["id", "question", "rag_best_similarity"]])
print("Unanswerable but answered:")
display(r[(~r["answerable"]) & (~r["rag_abstained"])][["id", "question", "rag_answer"]])

In [ ]:
# Optional ablation: Recall@k for different k (retrieval only, no generation)
qs = evaluate.load_questions("eval_questions.csv")
qs = qs[qs["answerable"]]
rows = []
for k in [1, 2, 3, 5, 8]:
    hit = []
    for _, q in qs.iterrows():
        a = assistant.index.search(q["question"], k)
        srcs = [{"source": h["chunk"].source, "page_start": h["chunk"].page_start,
                 "page_end": h["chunk"].page_end} for h in a]
        hit.append(any(evaluate._gold_hit(s, q["source_file"],
                                          evaluate.parse_pages(q["source_pages"])) for s in srcs))
    rows.append({"k": k, "recall@k": round(sum(hit) / len(hit), 3)})
pd.DataFrame(rows)

## 8. Reproducibility record (§5.7)

In [ ]:
import sys, json, torch, transformers, sentence_transformers, fitz
record = {"python": sys.version.split()[0], "torch": torch.__version__,
          "transformers": transformers.__version__,
          "sentence_transformers": sentence_transformers.__version__,
          "pymupdf": fitz.__doc__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
          "config": vars(config), "documents": index.documents}
with open("run_record.json", "w") as f:
    json.dump(record, f, indent=2, default=str)
print(json.dumps(record, indent=2, default=str)[:2000])